# UrbanHub — Partie 3 : flux IoT, pollution urbaine (OpenAQ)

Ce notebook présente les analyses exploratoires du flux IoT et répond aux quatre questions métier de la Partie 3.

**Données.** API OpenAQ v3, 191 capteurs actifs répartis autour de Paris, Lyon, Marseille, Toulouse et Bordeaux (rayon de 25 km), cinq polluants : PM2.5, PM10, NO2, O3 et CO. Période analysée : du **22 septembre au 22 octobre 2026**, soit environ 103 000 mesures nettoyées.

**Pipeline** (architecture médaillon, dans MinIO) : `bronze/openaq` (mesures brutes) → `silver/pollution` (nettoyées) → `gold/pollution_horaire_ville` (moyenne horaire par ville et polluant).

**Nettoyage.** Jointure avec les métadonnées des capteurs, conversion des unités en µg/m³ (toutes les mesures l'étaient déjà), suppression des valeurs négatives, des valeurs aberrantes (bornes de plausibilité par polluant) et des doublons. Les valeurs de plus de 48 h renvoyées par l'API pour des capteurs muets sont ignorées.

**Méthode d'agrégation.** Moyenne horaire par capteur, puis moyenne (ou médiane) entre capteurs d'une même ville. On évite ainsi qu'un capteur qui émet plus souvent pèse davantage dans la moyenne.

*Les commentaires ci-dessous correspondent aux résultats obtenus sur cette période. Si les analyses sont relancées plus tard, les tableaux et les figures se mettent à jour, mais le texte doit être relu.*

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

def find_root():
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "outputs" / "partie3").exists():
            return d
    raise FileNotFoundError("outputs/partie3 introuvable : lance d'abord `python -m iot.analysis`.")

ROOT = find_root()
TAB = ROOT / "outputs" / "partie3" / "tables"
FIG = ROOT / "outputs" / "partie3" / "figures"

def show(name, width=950):
    path = FIG / name
    if path.exists():
        display(Image(filename=path, width=width))
    else:
        print(f"(figure absente : {name})")

print("Racine du projet :", ROOT)

## 0. Couverture des données

Avant d'interpréter, il faut savoir ce que les capteurs couvrent réellement.

In [ ]:
q1 = pd.read_csv(TAB / "q1_ranking_stations.csv")
cover = q1.groupby(["city", "pollutant"]).size().unstack(fill_value=0)
print("Nombre de stations retenues (au moins 72 h de mesures) par ville et polluant :")
display(cover)

La couverture est très inégale. Paris est la mieux équipée, notamment pour le NO2. Le **CO** n'est mesuré que par un à quatre capteurs, et pas du tout à Toulouse ni à Bordeaux : les résultats qui le concernent sont indicatifs et ne permettent aucune comparaison entre villes.

## 1. Quelles zones présentent les niveaux de pollution les plus élevés ?

On classe les stations par niveau moyen sur la période. Pour éviter qu'une station avec quelques mesures seulement n'apparaisse en tête, on ne retient que celles qui ont au moins 72 heures de données.

In [ ]:
top = q1.sort_values(["pollutant", "mean"], ascending=[True, False]).groupby("pollutant").head(3)
display(top[["pollutant", "city", "location_name", "mean", "p95", "n_hours"]].round(1))

In [ ]:
show("q1_top_stations.png")

In [ ]:
show("q1_carte_no2.png", 1100)
show("q1_carte_pm25.png", 1100)

**Lecture.**

- **NO2, PM10 et PM2.5** : la station *Paris Auto A1 – Saint-Denis* arrive en tête pour les trois polluants (NO2 ≈ 49 µg/m³, PM10 ≈ 37 µg/m³, PM2.5 ≈ 12,7 µg/m³). Son nom évoque une station proche d'un axe autoroutier, ce qui est cohérent avec une pollution d'origine routière. Cette lecture repose sur le nom de la station : le type de station (trafic, fond urbain, périurbain) n'a pas été vérifié dans les métadonnées d'OpenAQ.
- **CO** : *Lyon Périphérique* est la plus élevée (≈ 237 µg/m³), mais le CO reste très loin de la valeur guide de l'OMS (4 000 µg/m³), et ce classement repose sur très peu de capteurs.
- **O3** : le maximum est à *Sausset-les-Pins* (Marseille, ≈ 76 µg/m³), une station éloignée des grands axes. C'est un schéma attendu : près du trafic, le monoxyde d'azote consomme l'ozone, qui s'accumule donc davantage loin des sources.

Les cartes montrent que, à l'intérieur d'une même agglomération, les écarts entre stations sont importants : la ville seule ne suffit pas à décrire l'exposition.

## 2. Existe-t-il une variation journalière de la pollution ?

On calcule, pour chaque ville et chaque polluant, le niveau moyen par heure de la journée (heure locale, Europe/Paris).

In [ ]:
q2 = pd.read_csv(TAB / "q2_profil_horaire.csv")
peak = q2.loc[q2.groupby(["pollutant", "city"])["mean"].idxmax()]
print("Heure locale du maximum du profil moyen :")
display(peak.pivot(index="city", columns="pollutant", values="local_hour").astype("Int64"))
amp = q2.groupby(["pollutant", "city"])["mean"].agg(lambda s: s.max() / s.min()).unstack(0).round(2)
print("Amplitude du cycle (niveau maximum / niveau minimum de la journée) :")
display(amp)

In [ ]:
show("q2_profil_journalier.png", 1100)

**Lecture.**

- **Oui, la pollution varie nettement au cours de la journée**, et le profil dépend du polluant.
- **Ozone** : maximum en fin d'après-midi (17 h à Lyon, Marseille, Toulouse et Bordeaux, 19 h à Paris). C'est le comportement d'un polluant secondaire formé sous l'effet du rayonnement solaire.
- **NO2** : maximum à 10 h à Paris et à Marseille, ce qui est compatible avec le trafic du matin. À Lyon, Toulouse et Bordeaux, le maximum du profil moyen est à **22 h**. Ce résultat est moins intuitif. Hypothèses à tester avec la météo : une couche d'air stable en soirée qui piège les émissions du trafic et du chauffage, ou des pics ponctuels qui tirent la moyenne vers le haut (voir la question 4).
- **Particules (PM10, PM2.5)** : maximum en fin de matinée dans la plupart des villes (PM10 entre 10 h et 13 h), sauf à Toulouse pour les PM2.5 (23 h).
- Un profil moyen peut masquer un double pic (matin et soir) : le lire avec la figure.
- Le profil du CO (un seul capteur par ville dans certains cas) n'est pas exploitable.

## 3. Quels polluants dominent dans les villes étudiées ?

On compare le niveau de chaque polluant à la valeur guide journalière de l'OMS (2021) : PM2.5 : 15 µg/m³, PM10 : 45 µg/m³, NO2 : 25 µg/m³, O3 : 100 µg/m³ (maximum journalier de la moyenne sur 8 h), CO : 4 mg/m³. Le « polluant dominant » est celui dont le niveau moyen est le plus proche (ou le plus au-dessus) de sa valeur guide. Ces valeurs sont des recommandations sanitaires, plus strictes que les seuils réglementaires européens.

In [ ]:
q3 = pd.read_csv(TAB / "q3_polluants_vs_oms.csv")
print("Niveau moyen / valeur guide OMS (1 = à la valeur guide) :")
display(q3.pivot(index="city", columns="pollutant", values="ratio_to_who").round(2))
print("Part des jours au-dessus de la valeur guide (%) :")
display(q3.pivot(index="city", columns="pollutant", values="pct_days_above_who").round(0))
print("Polluant dominant par ville :")
display(q3[q3.dominant_in_city][["city", "pollutant", "ratio_to_who"]].round(2).reset_index(drop=True))

In [ ]:
show("q3_oms.png", 1100)

**Lecture.**

- **Le NO2 domine à Paris (≈ 1,01), Marseille (≈ 0,96) et Lyon (≈ 0,84)** : à Paris, le niveau moyen atteint la valeur guide. C'est un polluant lié au trafic, ce qui rejoint la lecture de la question 1.
- **L'ozone domine à Toulouse (≈ 0,84) et à Bordeaux (≈ 0,76)**, et il est proche du NO2 à Marseille (≈ 0,89).
- **Les particules restent bien en dessous des valeurs guides** (PM10 entre 0,29 et 0,41, PM2.5 entre 0,36 et 0,56), et le CO est négligeable (≤ 0,06).
- **Précaution** : l'ozone est comparé à un indicateur de pic (maximum sur 8 h), les autres polluants à une moyenne journalière. Les ratios ne sont donc pas strictement comparables entre polluants. De plus, la période (septembre-octobre) est défavorable aux particules (chauffage, hiver) et moins favorable à l'ozone qu'en plein été : le classement pourrait changer sur une autre saison.

## 4. Peut-on identifier des épisodes de pollution anormale ?

**Méthode.** Pour chaque ville et chaque polluant, on prend la médiane entre capteurs à chaque heure (en ne gardant que les heures où au moins 3 capteurs ont mesuré), puis on calcule l'écart à la valeur habituelle de cette heure de la journée. Cet écart est converti en z-score robuste (basé sur la médiane et l'écart absolu médian, peu sensible aux valeurs extrêmes). Une heure est dite anormale quand ce z-score dépasse 3,5. Un **épisode** regroupe au moins 3 heures anormales, séparées de 3 heures au plus.

In [ ]:
try:
    q4 = pd.read_csv(TAB / "q4_episodes.csv", parse_dates=["start", "end"])
except pd.errors.EmptyDataError:
    q4 = pd.DataFrame()
if q4.empty:
    print("Aucun épisode détecté sur la période.")
else:
    print(len(q4), "épisodes détectés. Les 10 plus intenses (heures en UTC) :")
    display(q4.head(10).round(1))
    print("Nombre d'épisodes par ville et polluant :")
    display(q4.groupby(["city", "pollutant"]).size().unstack(fill_value=0))

In [ ]:
show("q4_episodes.png", 1100)

**Lecture.**

- **Oui, on détecte des épisodes** : 23 sur la période. Parmi les dix plus intenses, huit concernent le **NO2**, presque tous entre le 22 et le 25 septembre, et débutent en soirée (19 h à 21 h UTC, soit 21 h à 23 h en heure locale). Ils touchent **Toulouse, Bordeaux et Marseille aux mêmes dates** : une cause commune (conditions météorologiques stables) est plausible, mais elle reste à vérifier avec la météo.
- **Le contrôle capteur par capteur est indispensable**, car un épisode peut venir d'une seule station :
  - *Lyon, PM10, 24 septembre vers 10 h UTC* : un capteur (Gerland) atteint 485 µg/m³, contre 112 pour le suivant et 26 à 66 pour les autres. L'épisode est surtout dû à cette station. Toutefois, une hausse plus modérée (50 à 60 µg/m³ contre 20 à 30 l'heure précédente) touche aussi la moitié des capteurs vers 8 h UTC : une hausse réelle mais modeste.
  - *Toulouse, NO2, soirée du 24 septembre* : les sept stations montent ensemble (70 à 130 µg/m³ vers 20 h-21 h UTC contre 10 à 20 en temps normal). Les stations d'Arcole et de l'Embouchure précèdent les autres, qui rattrapent à 22 h UTC. C'est un épisode urbain réel, non lié à un capteur isolé.
- **Un épisode statistique n'est pas un dépassement réglementaire.** Un z-score élevé sur une base très faible (ex. PM2.5 à Bordeaux mi-octobre) peut correspondre à des niveaux modestes en valeur absolue.
- La détection repose sur 30 jours de référence seulement : une période plus longue rendrait la valeur « habituelle » plus fiable.

## Limites

- **Durée** : un mois d'automne. Les conclusions saisonnières (ozone l'été, particules l'hiver) ne peuvent pas être tirées.
- **Couverture inégale** : très peu de capteurs de CO, aucun à Toulouse ni à Bordeaux. Les villes moyennes ont 7 à 10 capteurs par polluant, contre 30 pour le NO2 à Paris.
- **Type de station non vérifié** : les lectures « trafic » ou « fond » s'appuient sur les noms des stations.
- **Qualité des capteurs** : les valeurs sont celles publiées par OpenAQ, sans contrôle indépendant. Le nettoyage supprime les valeurs impossibles mais ne détecte pas tous les capteurs dérivants.
- **Pas de variable explicative** dans cette partie : les hypothèses sur le rôle de la météo ne sont pas testées ici.

## Ouverture vers la Partie 4 (analyse croisée)

La table `gold/pollution_horaire_ville` (ville × polluant × heure, avec `timestamp` en UTC et `date_local` / `hour_local` en heure de Paris) permet la jointure avec la météo et les vélos. Les hypothèses à tester :

1. Les épisodes de NO2 du soir (22 au 25 septembre) coïncident-ils avec un vent faible et une pression élevée ?
2. L'ozone de l'après-midi suit-il la température ?
3. Les particules baissent-elles avec les précipitations ?
4. L'usage des vélos diminue-t-il lors des pics de pollution, ou plus simplement avec la pluie ?

**Point de vigilance** : les données de pollution et de vélos couvrent septembre-octobre 2026. La jointure suppose une météo disponible sur cette même période.

## Conclusion

1. Le **NO2 est le polluant le plus proche de sa valeur guide** à Paris, Marseille et Lyon, avec un maximum aux stations proches des grands axes. L'ozone domine à Toulouse et à Bordeaux. Les particules restent nettement sous les valeurs guides sur la période.
2. Les profils journaliers sont **marqués et dépendent du polluant** : ozone en fin d'après-midi, NO2 et particules plutôt le matin, avec un maximum de NO2 tardif (22 h) dans trois villes qui reste à expliquer.
3. Des **épisodes anormaux** existent, surtout pour le NO2 en soirée entre le 22 et le 25 septembre. Le contrôle capteur par capteur distingue des épisodes urbains réels (Toulouse) d'artefacts dus à une station (Lyon).
4. Ces résultats sont **descriptifs** : sans la météo, on ne peut formuler que des hypothèses. C'est l'objet de la Partie 4.